In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from plotly.subplots import make_subplots
import torch
import umap
import plotly.graph_objects as go
from scipy.interpolate import griddata
from lightning_models.lightning_conformal_mlp import LightningConformalClassifier
import itertools
import plotly.graph_objects as go
from datamodules.bray_data_module import BrayDataModule
from datamodules.image_feature_data_module import ImageFeatureDataModule
from lightning_models.lightning_diffusion_mlp import LightningDiffusionClassifier

In [2]:
#datamodule = BrayDataModule()
# datamodule = ImageFeatureDataModule(
#     split_col='hier_split_0',
#     data_csv_path='/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/data/bray_dino/bray_dino_complete.csv',
#     mask_uncertain= False,
#     validation_split_val="cal"
# )
datamodule = ImageFeatureDataModule(
    split_col='hier_split_0',
    data_csv_path='/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/data/bray_cloome/Bray_CLOOME_complete.csv',
    mask_uncertain= False,
    validation_split_val="cal"
)
datamodule.setup("fit")
datamodule

--- Setting up data for stage: fit ---
Loading ImageFeatureDataset using Polars lazy frames for split='train'...
Found 512 feature columns and 30 MoA columns
Executing query and loading data...
Loaded 205092 samples for split='train'
Converting to numpy arrays...
Feature dimension: 512
Target dimension: 30
Loading ImageFeatureDataset using Polars lazy frames for split='cal'...
Found 512 feature columns and 30 MoA columns
Executing query and loading data...
Loaded 51272 samples for split='cal'
Converting to numpy arrays...
Feature dimension: 512
Target dimension: 30
Training set size: 205092 samples
Val set size: 51272 samples
--- Data setup complete ---


In [3]:
diff_ckpt_path = "/checkpoints_gridsearch_old/cloome/diffusion/last.ckpt"
diff_model = LightningDiffusionClassifier(
    size="sm",
    num_classes=30,
    embedding_dim=512,
    loss_pos_weight=5.21,
    alpha=0.05,
    lr=1e-4,
    residual=True,
    activation_fn=torch.nn.GELU,
)
diff_model.load_state_dict(
    torch.load(diff_ckpt_path, weights_only=True, map_location=torch.device("mps"))[
        "state_dict"
    ]
)
diff_model.eval()

LightningDiffusionClassifier(
  (model): DinoDiffusionClassifier(
    (input_norm): LayerNorm((542,), eps=1e-05, elementwise_affine=True)
    (fc1): Linear(in_features=542, out_features=512, bias=True)
    (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (fc2): Linear(in_features=512, out_features=256, bias=True)
    (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc3): Linear(in_features=256, out_features=128, bias=True)
    (ln3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (fc4): Linear(in_features=128, out_features=30, bias=True)
    (res_proj1): Linear(in_features=542, out_features=512, bias=True)
    (res_proj2): Linear(in_features=512, out_features=256, bias=True)
    (res_proj3): Linear(in_features=256, out_features=128, bias=True)
    (activation): GELU(approximate='none')
    (dropout1): Dropout(p=0.3, inplace=False)
    (dropout2): Dropout(p=0.44999999999999996, inplace=False)
  )
)

In [4]:
conf_ckpt_path = "/checkpoints_gridsearch_old/cloome/conformal/last.ckpt"
conf_model = LightningConformalClassifier(
    size="xlg",
    num_classes=30,
    embedding_dim=512,
    loss_pos_weight=5.21,
    alpha=0.05,
    lr=1e-5,
    residual=True,
    activation_fn=torch.nn.GELU,
)
conf_model.load_state_dict(
    torch.load(conf_ckpt_path, weights_only=True, map_location=torch.device("mps"))[
        "state_dict"
    ]
)
conf_model.eval()

LightningConformalClassifier(
  (model): MLPClassifierXlg(
    (fc1): Linear(in_features=512, out_features=4096, bias=True)
    (ln1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (fc2): Linear(in_features=4096, out_features=2048, bias=True)
    (ln2): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (fc3): Linear(in_features=2048, out_features=1024, bias=True)
    (ln3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (fc4): Linear(in_features=1024, out_features=512, bias=True)
    (ln4): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (fc5): Linear(in_features=512, out_features=256, bias=True)
    (ln5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (fc6): Linear(in_features=256, out_features=128, bias=True)
    (ln6): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (fc_out): Linear(in_features=128, out_features=30, bias=True)
    (res_proj1): Linear(in_features=512, out_features=4096, bias=True)
    (res_proj2): Line

In [5]:
train_x_s = []
train_y_s = []

train_diff_preds = []
train_conf_preds = []

# Separate counters for each class
train_class_1_count = 0  # y[0] == 1
train_class_0_count = 0  # y[0] == 0
target_per_class = 250  # 500 each for 50:50 split

for idx, batch in enumerate(datamodule.train_dataloader()):
    if (
        train_class_1_count >= target_per_class
        and train_class_0_count >= target_per_class
    ):
        break

    x, y, mask = batch

    # Process class 1 samples (y[0] == 1 and rest zeros)
    if train_class_1_count < target_per_class:
        valid_mask_1 = y[:, 0] == 1
        valid_indices_1 = torch.where(valid_mask_1)[0]

        if len(valid_indices_1) > 0:
            # Limit to remaining needed samples for class 1
            remaining_1 = target_per_class - train_class_1_count
            if len(valid_indices_1) > remaining_1:
                valid_indices_1 = valid_indices_1[:remaining_1]

            x_valid_1 = x[valid_indices_1]
            y_valid_1 = y[valid_indices_1]

            train_x_s.append(x_valid_1.detach().numpy())
            train_y_s.append(y_valid_1.detach().numpy())

            # Diffusion predictions
            diff_y_preds = []
            for n in range(10):
                diff_y_pred = diff_model.forward(x_valid_1, y_valid_1)
                diff_y_preds.append(diff_y_pred.detach().numpy())

            diff_y_preds = np.array(diff_y_preds)
            class_avg_preds = np.mean(diff_y_preds, axis=0)
            train_diff_preds.append(class_avg_preds)

            # Conformal predictions
            conf_y_pred = conf_model.forward(x_valid_1)
            train_conf_preds.append(np.array(conf_y_pred.detach().numpy()))

            train_class_1_count += len(valid_indices_1)

    # Process class 0 samples (y[0] == 0 and rest zeros)
    if train_class_0_count < target_per_class:
        valid_mask_0 = y[:, 0] == 0
        valid_indices_0 = torch.where(valid_mask_0)[0]

        if len(valid_indices_0) > 0:
            # Limit to remaining needed samples for class 0
            remaining_0 = target_per_class - train_class_0_count
            if len(valid_indices_0) > remaining_0:
                valid_indices_0 = valid_indices_0[:remaining_0]

            x_valid_0 = x[valid_indices_0]
            y_valid_0 = y[valid_indices_0]

            train_x_s.append(x_valid_0.detach().numpy())
            train_y_s.append(y_valid_0.detach().numpy())

            # Diffusion predictions
            diff_y_preds = []
            for n in range(10):
                diff_y_pred = diff_model.forward(x_valid_0, y_valid_0)
                diff_y_preds.append(diff_y_pred.detach().numpy())

            diff_y_preds = np.array(diff_y_preds)
            class_avg_preds = np.mean(diff_y_preds, axis=0)
            train_diff_preds.append(class_avg_preds)

            # Conformal predictions
            conf_y_pred = conf_model.forward(x_valid_0)
            train_conf_preds.append(np.array(conf_y_pred.detach().numpy()))

            train_class_0_count += len(valid_indices_0)

print(
    f"Collected {train_class_1_count} samples with y[0]=1, {train_class_0_count} samples with y[0]=0"
)

train_x_s = np.concatenate(train_x_s, axis=0)
train_y_s = np.concatenate(train_y_s, axis=0)
train_diff_preds = np.concatenate(train_diff_preds, axis=0)
train_conf_preds = np.concatenate(train_conf_preds, axis=0)

print(
    f"Final shapes: x_s={train_x_s.shape}, y_s={train_y_s.shape}, diff_preds={train_diff_preds.shape}, conf_preds={train_conf_preds.shape}"
)

/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Collected 250 samples with y[0]=1, 250 samples with y[0]=0
Final shapes: x_s=(500, 512), y_s=(500, 30), diff_preds=(500, 30), conf_preds=(500, 30)


In [6]:
datamodule.setup("fit")

--- Setting up data for stage: fit ---
Loading ImageFeatureDataset using Polars lazy frames for split='train'...
Found 512 feature columns and 30 MoA columns
Executing query and loading data...
Loaded 205092 samples for split='train'
Converting to numpy arrays...
Feature dimension: 512
Target dimension: 30
Loading ImageFeatureDataset using Polars lazy frames for split='cal'...
Found 512 feature columns and 30 MoA columns
Executing query and loading data...
Loaded 51272 samples for split='cal'
Converting to numpy arrays...
Feature dimension: 512
Target dimension: 30
Training set size: 205092 samples
Val set size: 51272 samples
--- Data setup complete ---


In [7]:
x_s = []
y_s = []

diff_preds = []
conf_preds = []

# Separate counters for each class
class_1_count = 0  # y[0] == 1
class_0_count = 0  # y[0] == 0
target_per_class = 250  # 500 each for 50:50 split

for idx, batch in enumerate(datamodule.train_dataloader()):
    if class_1_count >= target_per_class and class_0_count >= target_per_class:
        break

    x, y, mask = batch

    # Process class 1 samples (y[0] == 1 and rest zeros)
    if class_1_count < target_per_class:
        valid_mask_1 = y[:, 0] == 1
        valid_indices_1 = torch.where(valid_mask_1)[0]

        if len(valid_indices_1) > 0:
            # Limit to remaining needed samples for class 1
            remaining_1 = target_per_class - class_1_count
            if len(valid_indices_1) > remaining_1:
                valid_indices_1 = valid_indices_1[:remaining_1]

            x_valid_1 = x[valid_indices_1]
            y_valid_1 = y[valid_indices_1]

            x_s.append(x_valid_1.detach().numpy())
            y_s.append(y_valid_1.detach().numpy())

            # Diffusion predictions
            diff_y_preds = []
            for n in range(10):
                diff_y_pred = diff_model.forward(x_valid_1, y_valid_1)
                diff_y_preds.append(diff_y_pred.detach().numpy())

            diff_y_preds = np.array(diff_y_preds)
            class_avg_preds = np.mean(diff_y_preds, axis=0)
            diff_preds.append(class_avg_preds)

            # Conformal predictions
            conf_y_pred = conf_model.forward(x_valid_1)
            conf_preds.append(np.array(conf_y_pred.detach().numpy()))

            class_1_count += len(valid_indices_1)

    # Process class 0 samples (y[0] == 0 and rest zeros)
    if class_0_count < target_per_class:
        valid_mask_0 = y[:, 0] == 0
        valid_indices_0 = torch.where(valid_mask_0)[0]

        if len(valid_indices_0) > 0:
            # Limit to remaining needed samples for class 0
            remaining_0 = target_per_class - class_0_count
            if len(valid_indices_0) > remaining_0:
                valid_indices_0 = valid_indices_0[:remaining_0]

            x_valid_0 = x[valid_indices_0]
            y_valid_0 = y[valid_indices_0]

            x_s.append(x_valid_0.detach().numpy())
            y_s.append(y_valid_0.detach().numpy())

            # Diffusion predictions
            diff_y_preds = []
            for n in range(10):
                diff_y_pred = diff_model.forward(x_valid_0, y_valid_0)
                diff_y_preds.append(diff_y_pred.detach().numpy())

            diff_y_preds = np.array(diff_y_preds)
            class_avg_preds = np.mean(diff_y_preds, axis=0)
            diff_preds.append(class_avg_preds)

            # Conformal predictions
            conf_y_pred = conf_model.forward(x_valid_0)
            conf_preds.append(np.array(conf_y_pred.detach().numpy()))

            class_0_count += len(valid_indices_0)

print(
    f"Collected {class_1_count} samples with y[0]=1, {class_0_count} samples with y[0]=0"
)

x_s = np.concatenate(x_s, axis=0)
y_s = np.concatenate(y_s, axis=0)
diff_preds = np.concatenate(diff_preds, axis=0)
conf_preds = np.concatenate(conf_preds, axis=0)

print(
    f"Final shapes: x_s={x_s.shape}, y_s={y_s.shape}, diff_preds={diff_preds.shape}, conf_preds={conf_preds.shape}"
)

Collected 250 samples with y[0]=1, 250 samples with y[0]=0
Final shapes: x_s=(500, 512), y_s=(500, 30), diff_preds=(500, 30), conf_preds=(500, 30)


# Separate embedding space

In [8]:
umap_embedding = umap.UMAP().fit_transform(x_s)
train_umap_embedding = umap.UMAP().fit_transform(train_x_s)

In [9]:
# Get test coordinates and labels
x_test, y_test = umap_embedding[:, 0], umap_embedding[:, 1]
z_diff_test = diff_preds[:, 0]
z_conf_test = conf_preds[:, 0]
true_labels_test = y_s[:, 0]

# Get train coordinates and labels
x_train, y_train = train_umap_embedding[:, 0], train_umap_embedding[:, 1]
z_diff_train = train_diff_preds[:, 0]
z_conf_train = train_conf_preds[:, 0]
true_labels_train = train_y_s[:, 0]

# Create common interpolation grid for test data
grid_x_test, grid_y_test = np.meshgrid(
    np.linspace(x_test.min(), x_test.max(), 200),
    np.linspace(y_test.min(), y_test.max(), 200),
)

# Create common interpolation grid for train data
grid_x_train, grid_y_train = np.meshgrid(
    np.linspace(x_train.min(), x_train.max(), 200),
    np.linspace(y_train.min(), y_train.max(), 200),
)

# Interpolate test predictions onto grid
grid_z_diff_test = griddata(
    (x_test, y_test), z_diff_test, (grid_x_test, grid_y_test), method="cubic"
)
grid_z_conf_test = griddata(
    (x_test, y_test), z_conf_test, (grid_x_test, grid_y_test), method="cubic"
)

# Interpolate train predictions onto grid
grid_z_diff_train = griddata(
    (x_train, y_train), z_diff_train, (grid_x_train, grid_y_train), method="cubic"
)
grid_z_conf_train = griddata(
    (x_train, y_train), z_conf_train, (grid_x_train, grid_y_train), method="cubic"
)

# Calculate common color scale range across all data
z_min = min(
    z_diff_test.min(), z_conf_test.min(), z_diff_train.min(), z_conf_train.min()
)
z_max = max(
    z_diff_test.max(), z_conf_test.max(), z_diff_train.max(), z_conf_train.max()
)

# Create 2x2 subplots
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Test: Diffusion, 160K",
        "Test: MLP, 560K",
        "Train: Diffusion, 160K",
        "Train: MLP, 560K",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.1,
)

# ===== TOP ROW: TEST DATA =====

# Add test diffusion prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x_test[0],
        y=grid_y_test[:, 0],
        z=grid_z_diff_test,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Test Diffusion",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=0.46, len=0.4, y=0.75),  # Position colorbar for top left
    ),
    row=1,
    col=1,
)

# Add test diffusion scatter points
fig.add_trace(
    go.Scatter(
        x=x_test,
        y=y_test,
        mode="markers",
        marker=dict(
            color=true_labels_test,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Test True Labels",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Add test conformal prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x_test[0],
        y=grid_y_test[:, 0],
        z=grid_z_conf_test,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Test Conformal",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=1.02, len=0.4, y=0.75),  # Position colorbar for top right
    ),
    row=1,
    col=2,
)

# Add test conformal scatter points
fig.add_trace(
    go.Scatter(
        x=x_test,
        y=y_test,
        mode="markers",
        marker=dict(
            color=true_labels_test,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Test True Labels",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# ===== BOTTOM ROW: TRAIN DATA =====

# Add train diffusion prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x_train[0],
        y=grid_y_train[:, 0],
        z=grid_z_diff_train,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Train Diffusion",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=0.46, len=0.4, y=0.25),  # Position colorbar for bottom left
    ),
    row=2,
    col=1,
)

# Add train diffusion scatter points
fig.add_trace(
    go.Scatter(
        x=x_train,
        y=y_train,
        mode="markers",
        marker=dict(
            color=true_labels_train,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Train True Labels",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Add train conformal prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x_train[0],
        y=grid_y_train[:, 0],
        z=grid_z_conf_train,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Train Conformal",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=1.02, len=0.4, y=0.25),  # Position colorbar for bottom right
        showlegend=False,
    ),
    row=2,
    col=2,
)

# Add train conformal scatter points
fig.add_trace(
    go.Scatter(
        x=x_train,
        y=y_train,
        mode="markers",
        marker=dict(
            color=true_labels_train,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Train True Labels",
        showlegend=False,
    ),
    row=2,
    col=2,
)

# Update layout
fig.update_layout(
    title_text=f"Prediction Landscapes: Test vs Train Data (Test n={len(diff_preds)}, Train n={len(train_diff_preds)})",
    width=1600,
    height=1200,
    title_x=0.5,
    annotations=list(fig.layout.annotations)
    + [
        dict(
            text="Scatter points: Red = class 0, Blue = class 1",
            showarrow=False,
            xref="paper",
            yref="paper",
            x=0.5,
            y=-0.05,
            xanchor="center",
            yanchor="top",
            font=dict(size=12),
        )
    ],
)

# Update axes labels and ranges
# Test data ranges
x_range_test = [x_test.min(), x_test.max()]
y_range_test = [y_test.min(), y_test.max()]

# Train data ranges
x_range_train = [x_train.min(), x_train.max()]
y_range_train = [y_train.min(), y_train.max()]

# Top row (test data)
fig.update_xaxes(title_text="UMAP 1", range=x_range_test, row=1, col=1)
fig.update_xaxes(title_text="UMAP 1", range=x_range_test, row=1, col=2)
fig.update_yaxes(title_text="UMAP 2", range=y_range_test, row=1, col=1)
fig.update_yaxes(title_text="UMAP 2", range=y_range_test, row=1, col=2)

# Bottom row (train data)
fig.update_xaxes(title_text="UMAP 1", range=x_range_train, row=2, col=1)
fig.update_xaxes(title_text="UMAP 1", range=x_range_train, row=2, col=2)
fig.update_yaxes(title_text="UMAP 2", range=y_range_train, row=2, col=1)
fig.update_yaxes(title_text="UMAP 2", range=y_range_train, row=2, col=2)

fig.show()

In [10]:
x_s_combined = np.concatenate(
    [x_s, train_x_s], axis=0
)  # concatenate along rows (default)
umap_embedding_combined = umap.UMAP().fit_transform(x_s_combined)
umap_embedding_combined

array([[10.235487 , -1.1315063],
       [ 2.282019 ,  7.772574 ],
       [12.470807 ,  4.4372873],
       ...,
       [ 7.242353 , -0.9984776],
       [ 8.725425 ,  2.5965817],
       [ 8.853912 ,  3.1732566]], shape=(1000, 2), dtype=float32)

In [11]:
# Create combined embeddings
x_s_combined = np.concatenate([x_s, train_x_s])
umap_embedding_combined = umap.UMAP(random_state=42).fit_transform(x_s_combined)

# Split the combined embedding back into test and train portions
n_test = len(x_s)
n_train = len(train_x_s)

umap_embedding_test = umap_embedding_combined[:n_test]
umap_embedding_train = umap_embedding_combined[n_test:]

# Get test coordinates and labels
x_test, y_test = umap_embedding_test[:, 0], umap_embedding_test[:, 1]
z_diff_test = diff_preds[:, 0]
z_conf_test = conf_preds[:, 0]
true_labels_test = y_s[:, 0]

# Get train coordinates and labels
x_train, y_train = umap_embedding_train[:, 0], umap_embedding_train[:, 1]
z_diff_train = train_diff_preds[:, 0]
z_conf_train = train_conf_preds[:, 0]
true_labels_train = train_y_s[:, 0]

# Create common interpolation grid for the entire combined space
x_min_combined = umap_embedding_combined[:, 0].min()
x_max_combined = umap_embedding_combined[:, 0].max()
y_min_combined = umap_embedding_combined[:, 1].min()
y_max_combined = umap_embedding_combined[:, 1].max()

grid_x, grid_y = np.meshgrid(
    np.linspace(x_min_combined, x_max_combined, 200),
    np.linspace(y_min_combined, y_max_combined, 200),
)

# Interpolate predictions onto the common grid
grid_z_diff_test = griddata(
    (x_test, y_test), z_diff_test, (grid_x, grid_y), method="cubic"
)
grid_z_conf_test = griddata(
    (x_test, y_test), z_conf_test, (grid_x, grid_y), method="cubic"
)
grid_z_diff_train = griddata(
    (x_train, y_train), z_diff_train, (grid_x, grid_y), method="cubic"
)
grid_z_conf_train = griddata(
    (x_train, y_train), z_conf_train, (grid_x, grid_y), method="cubic"
)

# Calculate common color scale range across all data
z_min = min(
    z_diff_test.min(), z_conf_test.min(), z_diff_train.min(), z_conf_train.min()
)
z_max = max(
    z_diff_test.max(), z_conf_test.max(), z_diff_train.max(), z_conf_train.max()
)

# Create 2x2 subplots
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Test: Diffusion, 10K",
        "Test: MLP, 41MK",
        "Train: Diffusion, 10K",
        "Train: MLP, 41MK",
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.1,
)

# ===== TOP ROW: TEST DATA =====

# Add test diffusion prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x[0],
        y=grid_y[:, 0],
        z=grid_z_diff_test,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Test Diffusion",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=0.46, len=0.4, y=0.75),
    ),
    row=1,
    col=1,
)

# Add test diffusion scatter points
fig.add_trace(
    go.Scatter(
        x=x_test,
        y=y_test,
        mode="markers",
        marker=dict(
            color=true_labels_test,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Test True Labels",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Add test conformal prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x[0],
        y=grid_y[:, 0],
        z=grid_z_conf_test,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Test Conformal",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=1.02, len=0.4, y=0.75),
    ),
    row=1,
    col=2,
)

# Add test conformal scatter points
fig.add_trace(
    go.Scatter(
        x=x_test,
        y=y_test,
        mode="markers",
        marker=dict(
            color=true_labels_test,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Test True Labels",
        showlegend=False,
    ),
    row=1,
    col=2,
)

# ===== BOTTOM ROW: TRAIN DATA =====

# Add train diffusion prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x[0],
        y=grid_y[:, 0],
        z=grid_z_diff_train,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Train Diffusion",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=0.46, len=0.4, y=0.25),
    ),
    row=2,
    col=1,
)

# Add train diffusion scatter points
fig.add_trace(
    go.Scatter(
        x=x_train,
        y=y_train,
        mode="markers",
        marker=dict(
            color=true_labels_train,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Train True Labels",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Add train conformal prediction contour
fig.add_trace(
    go.Contour(
        x=grid_x[0],
        y=grid_y[:, 0],
        z=grid_z_conf_train,
        colorscale="Viridis",
        showscale=True,
        opacity=0.8,
        name="Train Conformal",
        zmin=z_min,
        zmax=z_max,
        colorbar=dict(x=1.02, len=0.4, y=0.25),
    ),
    row=2,
    col=2,
)

# Add train conformal scatter points
fig.add_trace(
    go.Scatter(
        x=x_train,
        y=y_train,
        mode="markers",
        marker=dict(
            color=true_labels_train,
            colorscale=[[0, "red"], [1, "blue"]],
            size=4,
            line=dict(width=0.5, color="white"),
            showscale=False,
        ),
        name="Train True Labels",
        showlegend=False,
    ),
    row=2,
    col=2,
)

# Update layout
fig.update_layout(
    title_text=f"Prediction Landscapes: Combined UMAP Space (Test n={len(diff_preds)}, Train n={len(train_diff_preds)})",
    width=1600,
    height=1200,
    title_x=0.5,
    annotations=list(fig.layout.annotations)
    + [
        dict(
            text="Scatter points: Red = class 0, Blue = class 1 | Combined UMAP embedding space",
            showarrow=False,
            xref="paper",
            yref="paper",
            x=0.5,
            y=-0.05,
            xanchor="center",
            yanchor="top",
            font=dict(size=12),
        )
    ],
)

# Use the same axis ranges for all subplots (combined UMAP space)
x_range_combined = [x_min_combined, x_max_combined]
y_range_combined = [y_min_combined, y_max_combined]

# Apply common ranges to all subplots
fig.update_xaxes(title_text="UMAP 1", range=x_range_combined, row=1, col=1)
fig.update_xaxes(title_text="UMAP 1", range=x_range_combined, row=1, col=2)
fig.update_xaxes(title_text="UMAP 1", range=x_range_combined, row=2, col=1)
fig.update_xaxes(title_text="UMAP 1", range=x_range_combined, row=2, col=2)

fig.update_yaxes(title_text="UMAP 2", range=y_range_combined, row=1, col=1)
fig.update_yaxes(title_text="UMAP 2", range=y_range_combined, row=1, col=2)
fig.update_yaxes(title_text="UMAP 2", range=y_range_combined, row=2, col=1)
fig.update_yaxes(title_text="UMAP 2", range=y_range_combined, row=2, col=2)

fig.show()

/Users/jkosciukiewicz/Developer/UJ/Projects/HCS-DFC/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [12]:
fig.write_html('./shared.html')